# Lab 12: Token Management, Costs, and Quotas (Gemini 3.1)

Moving from prototype to production requires strict control over your budget and API limits. In this lab, we will cover how to count tokens, estimate costs, and handle errors (429 and 503 errors).

## Objectives
1. Count tokens for text and multimodal inputs using `count_tokens`.
2. Implement a **Dynamic Cost Estimator**.
3. Handle **Errors** with exponential backoff logic.
4. Optimize context windows via **History Truncation**.

In [ ]:
import os
import time

import httpx
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODELS = {
    "flash": "gemini-3.1-flash-lite-preview",
    "pro": "gemini-3.1-pro-preview"
}

print("Client initialized for cost monitoring.")

## 1. Accurate Token Counting

Before sending a request, you should always check how many tokens it contains to stay within your plan's limits.

In [ ]:
def audit_tokens(model_name, contents):
    token_response = client.models.count_tokens(
        model=model_name,
        contents=contents
    )
    print(f"--- Token Audit for {model_name} ---")
    print(f"Total Tokens: {token_response.total_tokens}")
    return token_response.total_tokens

# Test with a simple prompt
prompt = "Explain the concept of tokenization in LLMs."
audit_tokens(MODELS['flash'], prompt)

# Test with multimodal input (Image tokens are calculated based on dimensions)
image_url = "https://storage.googleapis.com/generativeai-downloads/images/scones.jpg"
image_data = httpx.get(image_url).content
multimodal_content = [
    types.Part.from_bytes(data=image_data, mime_type="image/jpeg"),
    "Describe this image."
]

audit_tokens(MODELS['flash'], multimodal_content)

## 2. Real-time Cost Estimation

We can calculate the real cost of a generation by inspecting the `usage_metadata` field in the response.

In [ ]:
PRICING = {
    MODELS['flash']: {"input": 0.10, "output": 0.30}, # Per 1M tokens
    MODELS['pro']: {"input": 1.25, "output": 3.75}
}

def get_cost(model_name, usage):
    rates = PRICING[model_name]
    input_cost = (usage.prompt_token_count / 1_000_000) * rates['input']
    output_cost = (usage.candidates_token_count / 1_000_000) * rates['output']
    return input_cost + output_cost

response = client.models.generate_content(
    model=MODELS['flash'],
    contents="Write a long poem about Python programming."
)

cost = get_cost(MODELS['flash'], response.usage_metadata)
print(f"Request cost: ${cost:.8f}")

## 3. Handling errors (429 Rate Limit Errors or 503 Unavailable Errors)

Free and Pay-as-you-go tiers have 'Requests Per Minute' (RPM) limits. If you exceed them, the API returns a 429 error. API can also be temporarily unavailable and return a 503 error. We implement a retry mechanism.

In [ ]:
def call_with_retry(prompt, max_retries=3):
    for i in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODELS['flash'],
                contents=prompt
            )
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e) or "503" in str(e):
                wait_time = (2 ** i) + 1 # Exponential backoff
                print(f"Error occured. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                raise e
    raise Exception("Max retries exceeded")

resp = call_with_retry("Hello, are you there?")
print("Response received successfully.")

## 4. Context Optimization: History Truncation

In long chat sessions, the cost increases linearly with each message. To save money, we can truncate the history manually while keeping the last few messages.

In [ ]:
chat = client.chats.create(model=MODELS['flash'])

# Simulate a very long chat session
for i in range(10):
    chat.send_message(f"Message {i}: Tell me a fact.")

print(f"History size: {len(chat.get_history())} messages.")
tokens_before = client.models.count_tokens(
    model=MODELS['flash'], contents=chat.get_history()
).total_tokens
print(f"Tokens in history: {tokens_before}")

# TRUNCATE: Keep only the last 4 messages (2 user + 2 model)
# get_history() returns a copy, so we slice it and recreate the chat
truncated_history = chat.get_history()[-4:]
chat = client.chats.create(model=MODELS['flash'], history=truncated_history)

tokens_after = client.models.count_tokens(
    model=MODELS['flash'], contents=chat.get_history()
).total_tokens

print(f"\nHistory size after truncation: {len(chat.get_history())}")
print(f"Tokens after truncation: {tokens_after}")
print(f"Savings: {tokens_before - tokens_after} tokens.")

## Summary

In this lab, you learned how to manage the financial and operational aspects of the Gemini API:
1. **Anticipation**: Using `count_tokens` to pre-audit requests.
2. **Budgeting**: Calculating real costs from usage metadata.
3. **Resilience**: Implementing backoff strategies when receiving errors from API.
4. **Efficiency**: Truncating history to keep context windows (and bills) small.